In [2]:
# %%
import os
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping

import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, html, dcc, dash_table, Input, Output
from dash.exceptions import PreventUpdate

warnings.filterwarnings("ignore")

# Reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("Setup complete.")

TensorFlow version: 2.20.0
Setup complete.


In [3]:
# %%
# ==========================================
# LOAD NF-UNSW-NB15-v3 DATASET
# ==========================================

file_path = r"NF-UNSW-NB15-v3.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Dataset not found: {file_path}\n"
        "Place NF-UNSW-NB15-v3.csv in the same folder as this notebook."
    )

print("Loading dataset...")

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nAttack distribution:")
print(df["Attack"].value_counts(dropna=False))

Loading dataset...
Dataset loaded successfully.
Shape: (2365424, 55)

Columns:
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_

In [4]:
df.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1,45870,149.171.126.1,53,17,5.0,130,2,...,0,0,0,0,0,0,0,0,0,Benign


In [ ]:
# Creating of binary labels
#Benign → Normal → 0
#Everything else → Attack → 1

In [3]:

# ==========================================
# CREATE BINARY TARGET
# ==========================================

df["label"] = np.where(
    df["Attack"].astype(str).str.strip().str.lower().eq("benign"),
    "Normal",
    "Attack"
)

# Explicit encoding:
# Normal = 0
# Attack = 1

df["label_enc"] = df["label"].map({
    "Normal": 0,
    "Attack": 1
}).astype("int8")

print("Binary label distribution:")
print(df["label"].value_counts())

print("\nPercentage distribution:")
print(
    (df["label"].value_counts(normalize=True) * 100).round(2)
)

print("\nEncoded distribution:")
print(df["label_enc"].value_counts())

Binary label distribution:
label
Normal    2237731
Attack     127693
Name: count, dtype: int64

Percentage distribution:
label
Normal    94.6
Attack     5.4
Name: proportion, dtype: float64

Encoded distribution:
label_enc
0    2237731
1     127693
Name: count, dtype: int64


In [4]:
# %%
# ==========================================
# DASHBOARD-COMPATIBLE COLUMNS
# ==========================================

# Convert NetFlow start time to datetime
df["timestamp"] = pd.to_datetime(
    df["FLOW_START_MILLISECONDS"],
    unit="ms",
    errors="coerce"
)

# Human-readable aliases used by the existing dashboard
df["src_ip"] = df["IPV4_SRC_ADDR"].astype(str)
df["dst_ip"] = df["IPV4_DST_ADDR"].astype(str)

df["src_port"] = pd.to_numeric(
    df["L4_SRC_PORT"],
    errors="coerce"
).fillna(0)

df["dst_port"] = pd.to_numeric(
    df["L4_DST_PORT"],
    errors="coerce"
).fillna(0)

df["pkt_count"] = (
    pd.to_numeric(df["IN_PKTS"], errors="coerce").fillna(0)
    +
    pd.to_numeric(df["OUT_PKTS"], errors="coerce").fillna(0)
)

df["byte_count"] = (
    pd.to_numeric(df["IN_BYTES"], errors="coerce").fillna(0)
    +
    pd.to_numeric(df["OUT_BYTES"], errors="coerce").fillna(0)
)

# Dataset duration is in milliseconds
df["duration"] = (
    pd.to_numeric(
        df["FLOW_DURATION_MILLISECONDS"],
        errors="coerce"
    ).fillna(0) / 1000.0
)

# Convert protocol number into readable name for dashboard
protocol_map = {
    6: "TCP",
    17: "UDP",
    1: "ICMP"
}

protocol_numeric = pd.to_numeric(
    df["PROTOCOL"],
    errors="coerce"
).fillna(0).astype(int)

df["protocol"] = (
    protocol_numeric
    .map(protocol_map)
    .fillna(protocol_numeric.astype(str))
)

# Packet rate
safe_duration = df["duration"].clip(lower=0.001)

df["pkt_rate"] = (
    df["pkt_count"] / safe_duration
)

# Replace problematic values
df["pkt_rate"] = (
    df["pkt_rate"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(
    df[
        [
            "timestamp",
            "src_ip",
            "dst_ip",
            "src_port",
            "dst_port",
            "protocol",
            "pkt_count",
            "byte_count",
            "duration",
            "pkt_rate",
            "label"
        ]
    ].head()
)

                timestamp      src_ip         dst_ip  src_port  dst_port  \
0 2015-02-18 06:49:53.040  59.166.0.2  149.171.126.3      4894        53   
1 2015-02-18 06:49:52.744  59.166.0.4  149.171.126.6     52671     31992   
2 2015-02-18 06:49:50.649  59.166.0.0  149.171.126.9     47290      6881   
3 2015-02-18 06:49:53.145  59.166.0.8  149.171.126.7     43310        53   
4 2015-02-18 06:49:53.239  59.166.0.1  149.171.126.1     45870        53   

  protocol  pkt_count  byte_count  duration     pkt_rate   label  
0      UDP          4         324     0.002  2000.000000  Normal  
1      TCP         56        7680     0.335   167.164179  Normal  
2      TCP        676      561878     2.460   274.796748  Normal  
3      UDP          4         324     0.001  4000.000000  Normal  
4      UDP          4         292     0.001  4000.000000  Normal  


In [5]:
# %%
# ==========================================
# FEATURE SELECTION
# ==========================================

feature_cols = [
    "L4_SRC_PORT",
    "L4_DST_PORT",
    "PROTOCOL",
    "L7_PROTO",

    "IN_BYTES",
    "IN_PKTS",
    "OUT_BYTES",
    "OUT_PKTS",

    "TCP_FLAGS",
    "CLIENT_TCP_FLAGS",
    "SERVER_TCP_FLAGS",

    "FLOW_DURATION_MILLISECONDS",
    "DURATION_IN",
    "DURATION_OUT",

    "MIN_TTL",
    "MAX_TTL",

    "LONGEST_FLOW_PKT",
    "SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN",
    "MAX_IP_PKT_LEN",

    "SRC_TO_DST_SECOND_BYTES",
    "DST_TO_SRC_SECOND_BYTES",

    "RETRANSMITTED_IN_BYTES",
    "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES",
    "RETRANSMITTED_OUT_PKTS",

    "SRC_TO_DST_AVG_THROUGHPUT",
    "DST_TO_SRC_AVG_THROUGHPUT",

    "NUM_PKTS_UP_TO_128_BYTES",
    "NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES",
    "NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES",

    "TCP_WIN_MAX_IN",
    "TCP_WIN_MAX_OUT",

    "ICMP_TYPE",
    "ICMP_IPV4_TYPE",

    "DNS_QUERY_ID",
    "DNS_QUERY_TYPE",
    "DNS_TTL_ANSWER",

    "FTP_COMMAND_RET_CODE",

    "SRC_TO_DST_IAT_MIN",
    "SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_AVG",
    "SRC_TO_DST_IAT_STDDEV",

    "DST_TO_SRC_IAT_MIN",
    "DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_AVG",
    "DST_TO_SRC_IAT_STDDEV"
]

X = df[feature_cols].copy()

# Force numeric
for col in feature_cols:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

# Reduce RAM usage
X = X.astype(np.float32)

y = df["label_enc"]

print("Feature matrix:", X.shape)
print("Target:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

print("\nMissing values:", X.isna().sum().sum())

Feature matrix: (2365424, 49)
Target: (2365424,)

Class distribution:
label_enc
0    2237731
1     127693
Name: count, dtype: int64

Missing values: 0


In [6]:
# %%
# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training samples: (1774068, 49)
Testing samples: (591356, 49)

Training class distribution:
label_enc
0    1678298
1      95770
Name: count, dtype: int64

Testing class distribution:
label_enc
0    559433
1     31923
Name: count, dtype: int64


In [7]:
# %%
# ==========================================
# FEATURE SCALING
# ==========================================

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Reduce memory
X_train_s = X_train_s.astype(np.float32)
X_test_s = X_test_s.astype(np.float32)

print("Scaling complete.")
print("X_train_s:", X_train_s.shape)
print("X_test_s:", X_test_s.shape)

Scaling complete.
X_train_s: (1774068, 49)
X_test_s: (591356, 49)


In [8]:
# %%
# ==========================================
# RANDOM FOREST
# ==========================================

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

print("Training Random Forest...")

rf.fit(X_train, y_train)

print("Random Forest training complete.")

y_pred = rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred)
rf_precision = precision_score(y_test, y_pred, pos_label=1)
rf_recall = recall_score(y_test, y_pred, pos_label=1)
rf_f1 = f1_score(y_test, y_pred, pos_label=1)

print("\nRandom Forest Results")
print("--------------------------------")
print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Normal", "Attack"],
        digits=4
    )
)

Training Random Forest...
Random Forest training complete.

Random Forest Results
--------------------------------
Accuracy : 0.999993235884983
Precision: 0.9999060297572435
Recall   : 0.9999686746233123
F1 Score : 0.9999373512091216

Classification Report:
              precision    recall  f1-score   support

      Normal     1.0000    1.0000    1.0000    559433
      Attack     0.9999    1.0000    0.9999     31923

    accuracy                         1.0000    591356
   macro avg     1.0000    1.0000    1.0000    591356
weighted avg     1.0000    1.0000    1.0000    591356



In [9]:
# %%
# ==========================================
# RANDOM FOREST PROBABILITIES
# ==========================================

print("RF classes:", rf.classes_)

rf_probs = rf.predict_proba(X_test)

print("Probability matrix:", rf_probs.shape)

attack_class_index = np.where(
    rf.classes_ == 1
)[0][0]

rf_attack_prob = rf_probs[:, attack_class_index]

print("Attack probability range:")
print("Min :", rf_attack_prob.min())
print("Max :", rf_attack_prob.max())
print("Mean:", rf_attack_prob.mean())

RF classes: [0 1]
Probability matrix: (591356, 2)
Attack probability range:
Min : 0.0
Max : 1.0
Mean: 0.05398986397364701


In [10]:
# %%
# ============================================================
# STRICTER VALIDATION: CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================
#
# Purpose:
# The main RF uses a random stratified split.
# This additional experiment tests whether the model can
# generalize to traffic occurring later in time.
#
# IMPORTANT:
# Variables use the prefix "chrono_" so the main model and
# dashboard are NOT affected.
# ============================================================

print("=" * 65)
print("CHRONOLOGICAL RANDOM FOREST VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Sort dataset chronologically
# ------------------------------------------------------------

chrono_order = np.argsort(
    df["FLOW_START_MILLISECONDS"].to_numpy()
)

chrono_X = X.iloc[chrono_order]
chrono_y = y.iloc[chrono_order]

# ------------------------------------------------------------
# 2. 75% earlier traffic = training
#    25% later traffic   = testing
# ------------------------------------------------------------

chrono_split = int(
    len(chrono_X) * 0.75
)

chrono_X_train = chrono_X.iloc[:chrono_split]
chrono_X_test = chrono_X.iloc[chrono_split:]

chrono_y_train = chrono_y.iloc[:chrono_split]
chrono_y_test = chrono_y.iloc[chrono_split:]

print("\nChronological split:")
print("Training samples:", len(chrono_X_train))
print("Testing samples :", len(chrono_X_test))

print("\nTraining class distribution:")
print(chrono_y_train.value_counts())

print("\nTesting class distribution:")
print(chrono_y_test.value_counts())

# ------------------------------------------------------------
# 3. Verify that both classes exist
# ------------------------------------------------------------

if chrono_y_train.nunique() < 2:
    raise ValueError(
        "Chronological training set contains only one class. "
        "A different time-based evaluation strategy is required."
    )

if chrono_y_test.nunique() < 2:
    raise ValueError(
        "Chronological test set contains only one class. "
        "A different time-based evaluation strategy is required."
    )

# ------------------------------------------------------------
# 4. Show actual time ranges
# ------------------------------------------------------------

chrono_train_indices = chrono_X_train.index
chrono_test_indices = chrono_X_test.index

train_start = df.loc[
    chrono_train_indices,
    "timestamp"
].min()

train_end = df.loc[
    chrono_train_indices,
    "timestamp"
].max()

test_start = df.loc[
    chrono_test_indices,
    "timestamp"
].min()

test_end = df.loc[
    chrono_test_indices,
    "timestamp"
].max()

print("\nTraining time range:")
print(train_start, "to", train_end)

print("\nTesting time range:")
print(test_start, "to", test_end)

# ------------------------------------------------------------
# 5. Train separate Random Forest
# ------------------------------------------------------------

chrono_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

print("\nTraining chronological Random Forest...")

chrono_rf.fit(
    chrono_X_train,
    chrono_y_train
)

print("Training complete.")

# ------------------------------------------------------------
# 6. Predict later traffic
# ------------------------------------------------------------

chrono_pred = chrono_rf.predict(
    chrono_X_test
)

# ------------------------------------------------------------
# 7. Metrics
# ------------------------------------------------------------

chrono_accuracy = accuracy_score(
    chrono_y_test,
    chrono_pred
)

chrono_precision = precision_score(
    chrono_y_test,
    chrono_pred,
    pos_label=1,
    zero_division=0
)

chrono_recall = recall_score(
    chrono_y_test,
    chrono_pred,
    pos_label=1,
    zero_division=0
)

chrono_f1 = f1_score(
    chrono_y_test,
    chrono_pred,
    pos_label=1,
    zero_division=0
)

print("\n" + "=" * 65)
print("CHRONOLOGICAL RF RESULTS")
print("=" * 65)

print("Accuracy :", chrono_accuracy)
print("Precision:", chrono_precision)
print("Recall   :", chrono_recall)
print("F1 Score :", chrono_f1)

print("\nClassification Report:")

print(
    classification_report(
        chrono_y_test,
        chrono_pred,
        target_names=[
            "Normal",
            "Attack"
        ],
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 8. Confusion Matrix
# ------------------------------------------------------------

chrono_cm = confusion_matrix(
    chrono_y_test,
    chrono_pred,
    labels=[0, 1]
)

print("\nConfusion Matrix:")
print(chrono_cm)

chrono_tn, chrono_fp, chrono_fn, chrono_tp = (
    chrono_cm.ravel()
)

print("\nTrue Negatives :", chrono_tn)
print("False Positives:", chrono_fp)
print("False Negatives:", chrono_fn)
print("True Positives :", chrono_tp)

# ------------------------------------------------------------
# 9. Feature importance
# ------------------------------------------------------------

chrono_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": chrono_rf.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

print("\nTop 15 Features:")
print(
    chrono_importance_df
    .head(15)
    .to_string(index=False)
)

CHRONOLOGICAL RANDOM FOREST VALIDATION

Chronological split:
Training samples: 1774068
Testing samples : 591356

Training class distribution:
label_enc
0    1697973
1      76095
Name: count, dtype: int64

Testing class distribution:
label_enc
0    539758
1     51598
Name: count, dtype: int64

Training time range:
2015-01-22 11:49:36.907000 to 2015-02-18 06:50:09.344000

Testing time range:
2015-02-18 06:50:09.344000 to 2015-02-18 12:29:24.927000

Training chronological Random Forest...
Training complete.

CHRONOLOGICAL RF RESULTS
Accuracy : 0.999993235884983
Precision: 0.9999612388077057
Recall   : 0.9999612388077057
F1 Score : 0.9999612388077057

Classification Report:
              precision    recall  f1-score   support

      Normal     1.0000    1.0000    1.0000    539758
      Attack     1.0000    1.0000    1.0000     51598

    accuracy                         1.0000    591356
   macro avg     1.0000    1.0000    1.0000    591356
weighted avg     1.0000    1.0000    1.0000    59

In [11]:
# %%
# ============================================================
# RANDOM SPLIT vs CHRONOLOGICAL SPLIT
# ============================================================

comparison_df = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Attack Precision",
        "Attack Recall",
        "Attack F1"
    ],

    "Random Split": [
        rf_accuracy,
        rf_precision,
        rf_recall,
        rf_f1
    ],

    "Chronological Split": [
        chrono_accuracy,
        chrono_precision,
        chrono_recall,
        chrono_f1
    ]
})

comparison_df["Difference"] = (
    comparison_df["Random Split"]
    -
    comparison_df["Chronological Split"]
)

print("=" * 75)
print("RANDOM vs CHRONOLOGICAL EVALUATION")
print("=" * 75)

print(
    comparison_df.to_string(
        index=False
    )
)

RANDOM vs CHRONOLOGICAL EVALUATION
          Metric  Random Split  Chronological Split  Difference
        Accuracy      0.999993             0.999993    0.000000
Attack Precision      0.999906             0.999961   -0.000055
   Attack Recall      0.999969             0.999961    0.000007
       Attack F1      0.999937             0.999961   -0.000024


In [12]:
# %%
# ==========================================
# AUTOENCODER TRAINING DATA
# ==========================================

# Select ONLY normal samples from TRAINING data
normal_train_mask = (y_train.values == 0)

X_normal_train_s = X_train_s[normal_train_mask]

print(
    "Normal training samples available:",
    X_normal_train_s.shape
)

# Split normal TRAINING samples into
# Autoencoder train and validation sets

Xn_train_s, Xn_val_s = train_test_split(
    X_normal_train_s,
    test_size=0.20,
    random_state=42
)

print(
    "AE training normal samples:",
    Xn_train_s.shape
)

print(
    "AE validation normal samples:",
    Xn_val_s.shape
)

Normal training samples available: (1678298, 49)
AE training normal samples: (1342638, 49)
AE validation normal samples: (335660, 49)


In [13]:
# %%
# ==========================================
# BUILD AUTOENCODER
# ==========================================

input_dim = Xn_train_s.shape[1]

encoding_dim = max(
    8,
    input_dim // 2
)

ae = models.Sequential([
    layers.Input(shape=(input_dim,)),

    layers.Dense(
        encoding_dim,
        activation="relu"
    ),

    layers.Dense(
        max(4, encoding_dim // 2),
        activation="relu"
    ),

    layers.Dense(
        encoding_dim,
        activation="relu"
    ),

    layers.Dense(
        input_dim,
        activation="linear"
    )
])

ae.compile(
    optimizer=optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

ae.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

print("\nTraining Autoencoder...")

history = ae.fit(
    Xn_train_s,
    Xn_train_s,

    validation_data=(
        Xn_val_s,
        Xn_val_s
    ),

    epochs=15,
    batch_size=512,

    callbacks=[early_stop],

    verbose=1
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 24)             │         1,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 24)             │           312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 49)             │         1,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,037 (11.86 KB)

 Trainable params: 3,037 (11.86 KB)

 Non-trainable params: 0 (0.00 B)


Training Autoencoder...
Epoch 1/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1225 - val_loss: 0.0487
Epoch 2/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0496 - val_loss: 0.0347
Epoch 3/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0389 - val_loss: 0.0274
Epoch 4/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0346 - val_loss: 0.0256
Epoch 5/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0328 - val_loss: 0.0236
Epoch 6/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0311 - val_loss: 0.0222
Epoch 7/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0292 - val_loss: 0.0210
Epoch 8/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0262 - val_loss: 0.0196
Epoch 9/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0228 - val_loss: 0.0191
Epoch 10/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0201 - val_loss: 0.0186
Epoch 11/15
2623/2623 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0186 - val_loss: 0.0180
Epoch

In [14]:
# %%
# ==========================================
# AUTOENCODER THRESHOLD
# ==========================================

recon_val = ae.predict(
    Xn_val_s,
    batch_size=2048,
    verbose=1
)

mse_val = np.mean(
    np.square(
        recon_val - Xn_val_s
    ),
    axis=1
)

# 99th percentile of NORMAL validation errors
# instead of blindly using mean + 3*std
thr = np.percentile(
    mse_val,
    99
)

print("\nAutoencoder threshold:", thr)

print(
    "Normal validation MSE mean:",
    mse_val.mean()
)

print(
    "Normal validation MSE median:",
    np.median(mse_val)
)

print(
    "Normal validation MSE max:",
    mse_val.max()
)

164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Autoencoder threshold: 0.06293521806597709
Normal validation MSE mean: 0.016301189
Normal validation MSE median: 0.0034893136
Normal validation MSE max: 1057.5059


In [15]:
# %%
# ==========================================
# AUTOENCODER TESTING
# ==========================================

recon_test = ae.predict(
    X_test_s,
    batch_size=2048,
    verbose=1
)

mse = np.mean(
    np.square(
        recon_test - X_test_s
    ),
    axis=1
)

# Explicit mapping:
#
# MSE <= threshold → Normal → 0
# MSE > threshold  → Attack → 1

ae_preds = (
    mse > thr
).astype(np.int8)

ae_accuracy = accuracy_score(
    y_test,
    ae_preds
)

ae_precision = precision_score(
    y_test,
    ae_preds,
    pos_label=1,
    zero_division=0
)

ae_recall = recall_score(
    y_test,
    ae_preds,
    pos_label=1,
    zero_division=0
)

ae_f1 = f1_score(
    y_test,
    ae_preds,
    pos_label=1,
    zero_division=0
)

print("\nAutoencoder Results")
print("--------------------------------")
print("Threshold :", thr)
print("Accuracy  :", ae_accuracy)
print("Precision :", ae_precision)
print("Recall    :", ae_recall)
print("F1 Score  :", ae_f1)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        ae_preds,
        target_names=[
            "Normal",
            "Attack"
        ],
        digits=4,
        zero_division=0
    )
)

289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Autoencoder Results
--------------------------------
Threshold : 0.06293521806597709
Accuracy  : 0.9905437672062176
Precision : 0.8509957610173559
Recall    : 0.999906023869937
F1 Score  : 0.9194607673695127

Classification Report:
              precision    recall  f1-score   support

      Normal     1.0000    0.9900    0.9950    559433
      Attack     0.8510    0.9999    0.9195     31923

    accuracy                         0.9905    591356
   macro avg     0.9255    0.9950    0.9572    591356
weighted avg     0.9920    0.9905    0.9909    591356



In [16]:
# %%
# ==========================================
# PREPARE DASHBOARD DATA
# ==========================================

# X_test retains the original dataframe indices
test_df = df.loc[X_test.index].copy()

# Keep exact same order as X_test
test_df = test_df.loc[X_test.index].copy()

# Predictions
test_df["rf_pred"] = y_pred

test_df["rf_prob"] = rf_attack_prob

test_df["ae_mse"] = mse

test_df["ae_pred"] = ae_preds

test_df["true_label"] = test_df["label"]

test_df["true_label_enc"] = y_test.values

# Normalize AE score to 0–1 for combined score
ae_score_normalized = (
    test_df["ae_mse"]
    /
    (test_df["ae_mse"].max() + 1e-9)
)

test_df["score_combined"] = (
    0.5 * test_df["rf_prob"]
    +
    0.5 * ae_score_normalized
)

# Ensure timestamp is datetime
test_df["timestamp"] = pd.to_datetime(
    test_df["timestamp"],
    errors="coerce"
)

# Remove invalid timestamps if any
test_df = test_df.dropna(
    subset=["timestamp"]
)

# Sort chronologically
test_df = (
    test_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    "Prepared visualization dataframe:",
    test_df.shape
)

print("\nColumns needed by dashboard:")

print(
    test_df[
        [
            "timestamp",
            "src_ip",
            "dst_ip",
            "src_port",
            "dst_port",
            "protocol",
            "pkt_count",
            "byte_count",
            "duration",
            "pkt_rate",
            "true_label",
            "rf_prob",
            "ae_mse",
            "score_combined"
        ]
    ].head()
)

Prepared visualization dataframe: (591356, 74)

Columns needed by dashboard:
                timestamp        src_ip          dst_ip  src_port  dst_port  \
0 2015-01-22 11:50:13.887  175.45.176.0  149.171.126.16     13284        80   
1 2015-01-22 11:50:14.036    59.166.0.5   149.171.126.2      6645        80   
2 2015-01-22 11:50:14.036    59.166.0.3   149.171.126.8     42587        25   
3 2015-01-22 11:50:14.322    59.166.0.0   149.171.126.5      3231        80   
4 2015-01-22 11:50:14.521    59.166.0.4   149.171.126.5     49346        53   

  protocol  pkt_count  byte_count  duration     pkt_rate true_label  rf_prob  \
0      TCP         20        1600     2.390     8.368201     Attack      1.0   
1      TCP         18        1876    29.268     0.615006     Normal      0.0   
2      TCP         94       40738    34.077     2.758459     Normal      0.0   
3      TCP         18        1876    28.789     0.625239     Normal      0.0   
4      UDP          4         296     0.001  400

In [17]:
# %%
from dash import Dash, html, dcc, dash_table, Input, Output
import dash

from dash.exceptions import PreventUpdate

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import confusion_matrix

In [18]:
# %%
# ==========================================
# DASHBOARD
# ==========================================

app = Dash(__name__)

app.layout = html.Div([

    html.H2(
        "AI-Powered Network Traffic Analyzer — Dashboard"
    ),

    html.Div([

        html.Div([

            html.Label(
                "Stream position (index)"
            ),

            dcc.Slider(
                id="pos-slider",
                min=0,
                max=max(
                    0,
                    len(test_df) - 1
                ),
                step=1,
                value=0
            )

        ], style={
            "width": "65%",
            "display": "inline-block",
            "padding": "10px"
        }),

        html.Div([

            html.Button(
                "Play Stream",
                id="play-btn",
                n_clicks=0
            ),

            html.Button(
                "Pause",
                id="pause-btn",
                n_clicks=0
            ),

            html.Button(
                "Jump to End",
                id="end-btn",
                n_clicks=0
            )

        ], style={
            "display": "inline-block",
            "verticalAlign": "top",
            "padding": "10px"
        })

    ]),

    html.Div(
        id="summary-cards",
        style={
            "display": "flex",
            "gap": "20px"
        }
    ),

    dcc.Graph(
        id="time-series-agg"
    ),

    dcc.Graph(
        id="scatter-score"
    ),

    html.H4(
        "Recent events"
    ),

    dash_table.DataTable(

        id="recent-table",

        columns=[
            {"name": c, "id": c}
            for c in [
                "timestamp",
                "src_ip",
                "dst_ip",
                "src_port",
                "dst_port",
                "protocol",
                "pkt_count",
                "byte_count",
                "duration",
                "true_label",
                "rf_prob",
                "ae_mse",
                "score_combined"
            ]
        ],

        data=test_df.head(10).to_dict(
            "records"
        ),

        page_size=10,

        style_table={
            "overflowX": "auto"
        }
    ),

    html.H4(
        "Confusion Matrix (RF)"
    ),

    dcc.Graph(
        id="conf-matrix"
    ),

    dcc.Interval(
        id="interval",
        interval=1500,
        n_intervals=0
    )

], style={
    "padding": "10px"
})


# ==========================================
# STREAMING CALLBACK
# ==========================================

@app.callback(

    Output(
        "pos-slider",
        "value"
    ),

    Input(
        "interval",
        "n_intervals"
    ),

    Input(
        "play-btn",
        "n_clicks"
    ),

    Input(
        "pause-btn",
        "n_clicks"
    ),

    Input(
        "end-btn",
        "n_clicks"
    ),

    Input(
        "pos-slider",
        "value"
    )
)

def auto_advance(
    n_intervals,
    play,
    pause,
    end,
    slider_val
):

    ctx = dash.callback_context

    if not ctx.triggered:
        return slider_val

    tid = (
        ctx.triggered[0]["prop_id"]
        .split(".")[0]
    )

    if tid == "play-btn":

        return min(
            len(test_df) - 1,
            slider_val + 1
        )

    if tid == "interval":

        if play > pause:

            return min(
                len(test_df) - 1,
                slider_val + 1
            )

        return slider_val

    if tid == "pause-btn":
        return slider_val

    if tid == "end-btn":
        return len(test_df) - 1

    return slider_val


# ==========================================
# DASHBOARD UPDATE CALLBACK
# ==========================================

@app.callback(

    Output(
        "recent-table",
        "data"
    ),

    Output(
        "time-series-agg",
        "figure"
    ),

    Output(
        "scatter-score",
        "figure"
    ),

    Output(
        "summary-cards",
        "children"
    ),

    Output(
        "conf-matrix",
        "figure"
    ),

    Input(
        "pos-slider",
        "value"
    )
)

def update_ui(pos):

    if pos is None:
        raise PreventUpdate

    # Recent 50-flow window
    window = test_df.iloc[
        max(0, pos - 50):
        pos + 1
    ].copy()

    # ==========================
    # SUMMARY
    # ==========================

    total = len(window)

    attacks = (
        window["true_label"]
        != "Normal"
    ).sum()

    # 1 = Attack
    rf_alerts = int(
        (window["rf_pred"] == 1)
        .sum()
    )

    ae_alerts = int(
        (window["ae_pred"] == 1)
        .sum()
    )

    cards = [

        html.Div([

            html.H3(
                "Window size"
            ),

            html.P(
                str(total)
            )

        ], style={
            "padding": "10px",
            "border": "1px solid #ddd",
            "borderRadius": "6px"
        }),

        html.Div([

            html.H3(
                "True Attacks"
            ),

            html.P(
                str(int(attacks))
            )

        ], style={
            "padding": "10px",
            "border": "1px solid #f88",
            "borderRadius": "6px",
            "backgroundColor": "#ffeeee"
        }),

        html.Div([

            html.H3(
                "RF Alerts"
            ),

            html.P(
                str(rf_alerts)
            )

        ], style={
            "padding": "10px",
            "border": "1px solid #f8d88c",
            "borderRadius": "6px"
        }),

        html.Div([

            html.H3(
                "AE Alerts"
            ),

            html.P(
                str(ae_alerts)
            )

        ], style={
            "padding": "10px",
            "border": "1px solid #8cf8d8",
            "borderRadius": "6px"
        })
    ]

    # ==========================
    # TIME SERIES
    # ==========================

    if window.empty:

        fig_ts = go.Figure()

    else:

        agg = (

            window
            .groupby(
                window[
                    "timestamp"
                ].dt.floor("s")
            )

            .agg(

                total=(
                    "timestamp",
                    "count"
                ),

                attacks=(
                    "true_label",
                    lambda s:
                    (s != "Normal").sum()
                )

            )

            .reset_index()
        )

        fig_ts = go.Figure()

        if not agg.empty:

            fig_ts.add_trace(

                go.Bar(
                    x=agg["timestamp"],
                    y=agg["total"],
                    name="Total"
                )
            )

            fig_ts.add_trace(

                go.Bar(
                    x=agg["timestamp"],
                    y=agg["attacks"],
                    name="True Attacks"
                )
            )

            fig_ts.update_layout(

                barmode="overlay",

                title=(
                    "Traffic count "
                    "(recent window)"
                )
            )

    # ==========================
    # SCATTER PLOT
    # ==========================

    if window.empty:

        fig_sc = go.Figure()

    else:

        fig_sc = px.scatter(

            window,

            x="pkt_rate",

            y="score_combined",

            color="true_label",

            hover_data=[
                "src_ip",
                "dst_ip",
                "rf_prob",
                "ae_mse"
            ]
        )

        fig_sc.update_layout(
            title=(
                "Score vs Packet rate "
                "(recent)"
            )
        )

    # ==========================
    # RF CONFUSION MATRIX
    # ==========================

    cm = confusion_matrix(

        test_df[
            "true_label_enc"
        ],

        test_df[
            "rf_pred"
        ],

        labels=[0, 1]
    )

    cm_fig = go.Figure(

        data=go.Heatmap(

            z=cm,

            x=[
                "Normal",
                "Attack"
            ],

            y=[
                "Normal",
                "Attack"
            ],

            colorscale="Reds"
        )
    )

    cm_fig.update_layout(

        title=(
            "Confusion matrix "
            "(RF) on test set"
        ),

        xaxis_title="Predicted",

        yaxis_title="True"
    )

    # ==========================
    # RECENT TABLE
    # ==========================

    recent_cols = [

        "timestamp",
        "src_ip",
        "dst_ip",
        "src_port",
        "dst_port",
        "protocol",
        "pkt_count",
        "byte_count",
        "duration",
        "true_label",
        "rf_prob",
        "ae_mse",
        "score_combined"
    ]

    recent_data = (

        window

        .sort_values(
            "timestamp",
            ascending=False
        )

        .head(20)[recent_cols]

        .to_dict(
            "records"
        )
    )

    return (
        recent_data,
        fig_ts,
        fig_sc,
        cards,
        cm_fig
    )


# ==========================================
# START DASHBOARD
# ==========================================

if __name__ == "__main__":

    app.run(
        debug=False,
        port=8051
    )

In [19]:
# Random Forest Feature Importance

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf.feature_importances_
}).sort_values(
    'Importance',
    ascending=False
)

print("Top 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

print("\n" + "="*50)

# Confusion Matrix Details

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

Top 20 Most Important Features:
                  Feature  Importance
                  MAX_TTL    0.215613
                  MIN_TTL    0.213527
           MIN_IP_PKT_LEN    0.154909
DST_TO_SRC_AVG_THROUGHPUT    0.075378
        SHORTEST_FLOW_PKT    0.062958
  SRC_TO_DST_SECOND_BYTES    0.062229
                OUT_BYTES    0.033414
         SERVER_TCP_FLAGS    0.025274
  DST_TO_SRC_SECOND_BYTES    0.023934
                TCP_FLAGS    0.016303
          TCP_WIN_MAX_OUT    0.015585
   RETRANSMITTED_OUT_PKTS    0.013463
 NUM_PKTS_UP_TO_128_BYTES    0.011211
                 OUT_PKTS    0.010236
       SRC_TO_DST_IAT_AVG    0.008392
           DNS_TTL_ANSWER    0.008361
    DST_TO_SRC_IAT_STDDEV    0.006646
                 L7_PROTO    0.005207
         LONGEST_FLOW_PKT    0.004088
           TCP_WIN_MAX_IN    0.003741

Confusion Matrix:
[[559430      3]
 [     1  31922]]

True Negatives : 559430
False Positives: 3
False Negatives: 1
True Positives : 31922
